# Strata on Colab: Qwen3.8-Flash-Next IQ3_S

Run cells in order on an A100 80 GB, high RAM Colab runtime. This is an experimental deployment: Strata's CUDA 12 package includes sm_80, but this exact Colab environment still needs an actual launch test. The model and MTP drafter are large downloads. The API key stays in the runtime environment and is never written to the artifact bundle.

In [ ]:
from pathlib import Path
import json, os, subprocess, secrets, sys
REPO_URL = "https://github.com/cao8011158/research-agent.git"
REPO_REF = "codex/colab-strata-experiments"  # editable; branch must be pushed first
ROOT = Path("/content/research-agent-" + secrets.token_hex(4))
ART = Path("/content/strata-artifacts")
ART.mkdir(exist_ok=True)
subprocess.run(["git", "clone", "--branch", REPO_REF, REPO_URL, str(ROOT)], check=True)
commit = subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True).strip()
(ART / "repository.json").write_text(json.dumps({"url": REPO_URL, "ref": REPO_REF, "commit": commit}, indent=2))
print("Repository commit:", commit)

## Code tests (Colab only)

Install the local package and run the full repository suite before installing Strata or downloading the model.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(ROOT) + "[test]"], check=True)
with (ART / "pytest.log").open("w") as log:
    result = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], cwd=ROOT, stdout=log, stderr=subprocess.STDOUT)
print((ART / "pytest.log").read_text()[-12000:])
result.check_returncode()

## Hardware and disk preflight

The gate checks host RAM separately from GPU memory and requires 100 GB free local scratch. It records the exact GPU, driver and compute capability. Do not proceed on a failed gate.

In [ ]:
preflight = subprocess.run([sys.executable, str(ROOT / "scripts/colab_preflight.py")], check=False)
preflight.check_returncode()
report = json.loads((ART / "preflight.json").read_text())
assert report["gate_passed"]
assert any(card["compute_capability"] == "8.0" and "A100" in card["name"] for card in report["gpus"]), "This experiment targets A100 sm_80 only"
print("A100 preflight passed; CUDA 12 package choice still needs verification in Strata plan")

## Install pinned Strata Terminal

Version 0.2.10 is pinned for reproducibility. The signed feed may select a separate Control or engine version; record what actually installs. Terminal download is small relative to model weights.

In [ ]:
import urllib.request
installer_url = "https://github.com/Niko1221/Strata/releases/download/v0.2.10/install.sh"
installer = Path("/content/strata-install.sh")
urllib.request.urlretrieve(installer_url, installer)
subprocess.run(["sh", str(installer)], check=True)
STRATA = str(Path.home() / ".local/bin/strata")
version = subprocess.check_output([STRATA, "--version"], text=True).strip()
(ART / "strata-terminal-version.txt").write_text(version + "\n")
print("Installed Terminal:", version)
assert "0.2.10" in version, "Pinned Terminal version did not install"

## Official launch plan

This cell has no model download. Inspect its printed model, drafter, files and selected CUDA engine before continuing. The launcher is the authority for the actual package selection; CUDA 13's release build omits sm_80.

In [ ]:
os.environ["STRATA_API_KEY"] = secrets.token_urlsafe(32)
LAUNCH = [STRATA, "launch", "--family", "qwen", "--model", "iq3_s",
          "--context", "32k", "--parallel", "1", "--vision", "off",
          "--drafting", "mtp", "--access", "local", "--port", "8080",
          "--data-dir", "/content/strata-data", "--api-key-env", "STRATA_API_KEY",
          "--output", "json"]
plan = subprocess.run(LAUNCH + ["--plan"], text=True, capture_output=True)
plan_text = (plan.stdout + "\n" + plan.stderr).replace(os.environ["STRATA_API_KEY"], "[REDACTED]")
(ART / "strata-plan.log").write_text(plan_text)
print(plan_text)
plan.check_returncode()
assert "iq3_s" in plan_text.lower(), "Plan did not name IQ3_S"
# The request contains --drafting mtp; actual choice is checked from /metrics after start.
# The plan may omit the engine variant. Reject an explicit CUDA 13 selection for sm_80.
assert "cuda13" not in plan_text.lower() and "cuda 13" not in plan_text.lower(), "Plan selected CUDA 13, which omits sm_80"

## Start model

This is the first heavy download. It can take a long time. If the plan or package compatibility is unclear, stop here and inspect it. Strata may silently choose another drafter if MTP is missing, so the request flag alone is not proof of the running choice.

In [ ]:
assert report["gate_passed"] and (ART / "strata-plan.log").exists()
started = subprocess.run(LAUNCH + ["--yes", "--no-tui"], text=True, capture_output=True)
start_text = (started.stdout + "\n" + started.stderr).replace(os.environ["STRATA_API_KEY"], "[REDACTED]")
(ART / "strata-start.log").write_text(start_text)
print(start_text[-12000:])
started.check_returncode()
# Actual drafter is verified from /metrics.engine.drafting by the next cell.
status = subprocess.run([STRATA, "launch", "--status", "--output", "json"], text=True, capture_output=True)
(ART / "strata-status.json").write_text((status.stdout + "\n" + status.stderr).replace(os.environ["STRATA_API_KEY"], "[REDACTED]"))
status.check_returncode()
print("Status saved:", ART / "strata-status.json")

## Real API smoke checks

Checks health, loaded model, properties, normal chat, a tool call and a continuation with the tool result. A model may decline the suggested tool; that remains a failed experiment rather than a fabricated success.

In [ ]:
smoke = subprocess.run([sys.executable, str(ROOT / "scripts/strata_smoke.py")], check=False)
smoke.check_returncode()
print("Smoke result saved:", ART / "smoke.json")

## Bundle diagnostics

Download the small JSON and log files for review. Model weights stay on local Colab scratch, not Drive. This cell does not run evaluation.

In [ ]:
import zipfile
bundle = Path("/content/strata-artifacts.zip")
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as zf:
    for file in ART.iterdir():
        if file.is_file():
            zf.write(file, file.name)
print(bundle)
from google.colab import files
files.download(str(bundle))

## Optional cleanup

Run this only after all intended API calls. It stops this local Strata launch and releases model memory.

In [ ]:
subprocess.run([STRATA, "launch", "--stop"], check=True)